In [1]:
import os
import geopandas as gpd
import pandas as pd
import numpy as np
import shapely.geometry as sg
import folium
import base64
import requests
import json 
import time
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from folium import CircleMarker

In [2]:
from xyzservices import TileProvider
from zwerfafval_detectie.utils_eval import read_annotations_folder


RD_CRS = "EPSG:28992"  # CRS code for the Dutch Rijksdriehoek coordinate system
LAT_LON_CRS = "EPSG:4326"  # CRS code for WGS84 latitude/longitude coordinate system

ams_tile_provider = TileProvider(
    name="Topografie, standaard visualisatie (WM)",
    url="https://t1.data.amsterdam.nl/topo_wm/{z}/{x}/{y}.png",
    attribution="data.amsterdam.nl",
)

DefaultAzureCredential failed to retrieve a token from the included credentials.
Attempted credentials:
	EnvironmentCredential: EnvironmentCredential authentication unavailable. Environment variables are not fully configured.
Visit https://aka.ms/azsdk/python/identity/environmentcredential/troubleshoot to troubleshoot this issue.
	WorkloadIdentityCredential: WorkloadIdentityCredential authentication unavailable. The workload options are not fully configured. See the troubleshooting guide for more information: https://aka.ms/azsdk/python/identity/workloadidentitycredential/troubleshoot. Missing required arguments: 'tenant_id', 'client_id', 'token_file_path'.
	ManagedIdentityCredential: ManagedIdentityCredential authentication unavailable, no response from the IMDS endpoint.
	SharedTokenCacheCredential: SharedTokenCacheCredential authentication unavailable. No accounts were found in the cache.
	VisualStudioCodeCredential: VisualStudioCodeCredential requires the 'azure-identity-broker' pa

In [ ]:
# if start from scratch, start with loading the model, predictions
model = "yolo26m_1920_v1-2_extra_250-2"
split = "train"

predictions_folder = f"../datasets/experiments/zwerfafval/predict/{model}/{split}"

categories = {
    0: "Zwerfafval (grof)",
    1: "Zwerfafval (fijn)"
}

confidence = 0.3 #confidence threshold for yolo26

In [ ]:
# Read predictions from the folder, create gdf
predictions_gdf = read_annotations_folder(folder_path=predictions_folder, categories=categories)
predictions_gdf["file_name"] = predictions_gdf["file_name"].str.replace(".txt", ".jpg")

_predictions_sorted = (
    predictions_gdf[predictions_gdf["confidence"] >= confidence]
    .set_index("file_name")
    .sort_index()
)

#create df with counts of each category per image
counts_df = (
    _predictions_sorted[["category"]]
    .replace(categories)
    .groupby(["file_name", "category"])
    .size()
    .unstack(fill_value=0)
)

In [ ]:
# Read metadata (gpkg) from the folder, create gdf
metadata_files = [
    "../datasets/experiments/zwerfafval/annotatieproject/inwinning_250514_selectie_300.gpkg",
    "../datasets/experiments/zwerfafval/annotatieproject/inwinning_260421_selectie_1000.gpkg"
]

metadata_gdf = pd.concat([
    gpd.read_file(metadata_file, layer=0)
    for metadata_file in metadata_files
]).set_index("file_name")

In [ ]:
# Merge counts_df with metadata_gdf to create a gdf with counts and geometry
counts_merged = gpd.GeoDataFrame(counts_df.join(metadata_gdf, how="left"))
counts_merged = counts_merged[["Zwerfafval (fijn)", "Zwerfafval (grof)", "geometry"]].to_crs(RD_CRS).dropna()

In [ ]:
#View merged counts, image and geometry
counts_merged

,Zwerfafval (fijn),Zwerfafval (grof),geometry
file_name,,,
20250514_194817_711097_000511.jpg,2,2,POINT (121449.197 485076.852)
20250514_194824_789533_000623.jpg,12,3,POINT (121451.399 485071.953)
20250514_194826_554034_000651.jpg,10,1,POINT (121452.572 485066.148)
20250514_194827_871855_000672.jpg,5,1,POINT (121453.264 485060.735)
20250514_194831_874718_000735.jpg,3,2,POINT (121454.223 485043.46)
...,...,...,...
20250514_200626_221499_017745.jpg,4,1,POINT (120926.994 485363.026)
20250514_200627_512295_017766.jpg,2,1,POINT (120927.609 485357.303)
20250514_200628_869633_017787.jpg,0,2,POINT (120928.096 485350.823)


In [ ]:
# Option: Fixed-radius dots, color = density
#------------------------------------------------


#data
points = counts_merged.copy()
points["total"] = points["Zwerfafval (fijn)"] + points["Zwerfafval (grof)"]
points = points[points["total"] > 0].copy()
points_wgs = points.to_crs("EPSG:4326")

#Color scale
cmap     = plt.get_cmap("YlOrRd")
log_vals = np.log1p(points_wgs["total"].values.astype(float))
norm     = mcolors.Normalize(vmin=log_vals.min(), vmax=log_vals.max())

def density_to_hex(val):
    return mcolors.to_hex(cmap(norm(np.log1p(val))))


#Create map
center = [points_wgs.geometry.y.mean(), points_wgs.geometry.x.mean()]

m = folium.Map(location=center, zoom_start=13, tiles=None)
folium.TileLayer(
    tiles="https://t1.data.amsterdam.nl/topo_wm/{z}/{x}/{y}.png",
    attr="data.amsterdam.nl",
    name="Amsterdam Topo",
).add_to(m)

for _, row in points_wgs.iterrows():
    total = int(row["total"])
    grof  = int(row["Zwerfafval (grof)"])
    fijn  = int(row["Zwerfafval (fijn)"])
    color = density_to_hex(total)

    CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=4,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.85,
        weight=0.8,
        tooltip=folium.Tooltip(
            f"""<div style="font-family:system-ui;font-size:12px">
              <b>Totaal:</b> {total} detecties<br>
              <b>Grof:</b> {grof} &nbsp;|&nbsp; <b>Fijn:</b> {fijn}
            </div>""",
            sticky=True,
        ),
    ).add_to(m)


legend_html = """
<div style="position:fixed;bottom:40px;left:20px;z-index:9999;
    background:white;border-radius:8px;padding:12px 16px;
    box-shadow:0 1px 6px rgba(0,0,0,.2);font-family:system-ui;font-size:12px">
  <div style="font-weight:600;margin-bottom:8px;color:#111">Detectiedichtheid</div>
  <div style="width:120px;height:14px;border-radius:3px;
    background:linear-gradient(to right,#ffffb2,#fecc5c,#fd8d3c,#f03b20,#bd0026)"></div>
  <div style="display:flex;justify-content:space-between;width:120px;margin-top:3px;color:#666">
    <span>Laag</span><span>Hoog</span>
  </div>
  <div style="margin-top:8px;color:#888;font-size:10px">Radius = vast (6px) | Kleur = log(fijn + grof)</div>
</div>"""
m.get_root().html.add_child(folium.Element(legend_html))


out_path = os.path.join(
    "../datasets/experiments/zwerfafval",
    f"heatmap_dots_{model}_{split}.html"
)
m.save(out_path)
print(f"Saved → {out_path}  ({len(points_wgs)} dots)")

Saved → ../datasets/experiments/zwerfafval/heatmap_dots_yolo26m_1920_v1-2_extra_250-2_train.html  (160 dots)


In [ ]:
# Option_final: Fixed-radius dots, two category layers, hover over point: show image, street name, & datetime
from folium.plugins import GroupedLayerControl
from datetime import datetime

# Data points and convert to WGS84 for mapping coordinates
points = counts_merged.copy()
points["total"] = points["Zwerfafval (fijn)"] + points["Zwerfafval (grof)"]
points = points[points["total"] > 0].copy()
points_wgs = points.to_crs("EPSG:4326")

# 2. Download official Amsterdam Streetnames LineStrings GeoJSON
print("Downloading Amsterdam street names GeoJSON...")
street_url = "https://maps.amsterdam.nl/open_geodata/geojson_lnglat.php?KAARTLAAG=STRAATNAMEN&THEMA=straatnamen"
response = requests.get(street_url)
streets_gdf = gpd.GeoDataFrame.from_features(response.json()["features"], crs="EPSG:4326")

# Use column 'STT_NAAM' which is used in Amsterdam's open data map layer
name_col = next((col for col in ['STT_NAAM', 'Naam', 'straatnaam', 'name', 'STRAATNAAM'] if col in streets_gdf.columns), streets_gdf.columns[0])
print(f"Using street name column: {name_col}")

# Couple each point to the closest street LineString using metric projection (EPSG:28992)
print("Coupling each point to the closest street line...")
points_m = points_wgs.to_crs("EPSG:28992")
streets_m = streets_gdf.to_crs("EPSG:28992")

joined = gpd.sjoin_nearest(points_m, streets_m, how="left", distance_col="dist_to_street")

# Assign the nearest found street name back to points_wgs
points_wgs["street"] = joined[name_col].values

# Image lookup indexing, avoid loading images from disk for every point, 
# (inwinning_250514_selectie_300, inwinning_250514_selectie_1000)
image_folders = [
    "/home/ding001/Zwerfafval-Detectie/datasets/experiments/zwerfafval/annotatieproject/inwinning_250514_selectie_300",
    "/home/ding001/Zwerfafval-Detectie/datasets/experiments/zwerfafval/annotatieproject/inwinning_250514_selectie_1000",
]

def build_image_index(folders):
    index = {}
    for folder in folders:
        if not os.path.isdir(folder):
            print(f"  ⚠ Folder not found: {folder}")
            continue
        for fname in os.listdir(folder):
            if fname.lower().endswith((".jpg", ".jpeg", ".png")):
                key = os.path.splitext(fname)[0]
                index[key] = os.path.join(folder, fname)
    return index

print("Indexing images...")
image_index = build_image_index(image_folders)
print(f"  Found {len(image_index)} images")

def get_image_b64(file_name):
    key = os.path.splitext(os.path.basename(file_name))[0]
    path = image_index.get(key)
    if path and os.path.isfile(path):
        with open(path, "rb") as f:
            data = base64.b64encode(f.read()).decode("utf-8")
        ext = os.path.splitext(path)[1].lower().replace(".jpg", ".jpeg")
        return f"data:image/{ext.strip('.')};base64,{data}"
    return None

# 5. Linear Color Scales per category
def make_norm_hex(series, cmap_name):
    cmap  = plt.get_cmap(cmap_name)
    min_v = float(series.values.min())
    max_v = float(series.values.max())
    if max_v == min_v:
        max_v = min_v + 1.0
    norm  = mcolors.Normalize(vmin=min_v, vmax=max_v)
    def to_hex(val):
        return mcolors.to_hex(cmap(norm(float(val))))
    return to_hex, int(min_v), int(max_v)

to_hex_grof, min_grof, max_grof = make_norm_hex(points_wgs["Zwerfafval (grof)"], "YlOrRd")
to_hex_fijn, min_fijn, max_fijn = make_norm_hex(points_wgs["Zwerfafval (fijn)"], "YlGnBu")

# 6. Create Map
center = [points_wgs.geometry.y.mean(), points_wgs.geometry.x.mean()]
m = folium.Map(location=center, zoom_start=13, tiles=None)
folium.TileLayer(
    tiles="https://t1.data.amsterdam.nl/topo_wm/{z}/{x}/{y}.png",
    attr="data.amsterdam.nl",
    name="Amsterdam Topo",
).add_to(m)

layer_grof = folium.FeatureGroup(name="Zwerfafval (grof)", show=True)
layer_fijn = folium.FeatureGroup(name="Zwerfafval (fijn)", show=True)

missing_images = 0

for idx, (file_name, row) in enumerate(points_wgs.iterrows()):
    lat   = row.geometry.y
    lon   = row.geometry.x
    grof  = int(row["Zwerfafval (grof)"])
    fijn  = int(row["Zwerfafval (fijn)"])
    total = int(row["total"])
    
    # Retrieve the dynamically mapped street name
    street = row.get("street", "Onbekende straat")
    if not isinstance(street, str) or not street.strip() or street == "nan":
        street = "Onbekende straat"
        
    # Check possible timestamp column names
    datetime_str = row.get("datetime", row.get("tijd", row.get("date", row.get("timestamp", "Onbekende tijd"))))

    # Show image (1080x720px) or (720x480), depending on screen size
    img_b64 = get_image_b64(file_name)
    if img_b64:
        img_html = f'<img src="{img_b64}" style="width:1080px;height:720px;object-fit:cover;border-radius:4px;display:block;margin-bottom:6px">'
    else:
        missing_images += 1
        img_html = '<div style="width:1080px;height:720px;background:#f0f0ee;border-radius:4px;display:flex;align-items:center;justify-content:center;color:#aaa;font-size:12px;margin-bottom:6px">Geen afbeelding</div>'

    fname_label = os.path.basename(file_name)

    def make_tooltip(count, color, category):
        return folium.Tooltip(
            f"""<div style="font-family:system-ui;font-size:12px;max-width:380px">
              {img_html}
              <div style="font-size:10px;color:#888;margin-bottom:4px;word-break:break-all"><b>Bestand:</b> {fname_label}</div>
              <div style="font-size:11px;color:#333;margin-bottom:2px">📍 <b>Straat:</b> {street}</div>
              <div style="font-size:11px;color:#333;margin-bottom:4px">🕒 <b>Tijdstip:</b> {datetime_str}</div>
              <div style="display:flex;align-items:center;gap:6px;margin-bottom:2px">
                <span style="display:inline-block;width:10px;height:10px;border-radius:2px;background:{color}"></span>
                <b>{category}:</b> {count}
              </div>
              <div style="color:#555"><b>Totaal:</b> {total} &nbsp;|&nbsp;
                <b>Grof:</b> {grof} &nbsp;|&nbsp; <b>Fijn:</b> {fijn}
              </div>
            </div>""",
            sticky=True,
        )

    if grof > 0:
        color_grof = to_hex_grof(grof)
        CircleMarker(
            location=[lat, lon],
            radius=5,
            color=color_grof,
            fill=True,
            fill_color=color_grof,
            fill_opacity=0.7,
            weight=0.8,
            tooltip=make_tooltip(grof, color_grof, "Grof"),
        ).add_to(layer_grof)

    if fijn > 0:
        color_fijn = to_hex_fijn(fijn)
        CircleMarker(
            location=[lat + 0.00003, lon + 0.00003],
            radius=5,
            color=color_fijn,
            fill=True,
            fill_color=color_fijn,
            fill_opacity=0.7,
            weight=0.8,
            tooltip=make_tooltip(fijn, color_fijn, "Fijn"),
        ).add_to(layer_fijn)

layer_grof.add_to(m)
layer_fijn.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)

print(f"  {missing_images} points had no matching image")


# Linear Scale Legend of both categories
legend_html = f"""
<div style="position:fixed;bottom:40px;left:20px;z-index:9999;
    background:white;border-radius:8px;padding:12px 16px;
    box-shadow:0 1px 6px rgba(0,0,0,.2);font-family:system-ui;font-size:12px;min-width:180px">
  <div style="font-weight:600;margin-bottom:10px;color:#111">Detectiedichtheid (Lineair)</div>

  <div style="margin-bottom:6px;font-size:11px;color:#444;font-weight:600">Grof</div>
  <div style="width:150px;height:12px;border-radius:3px;
    background:linear-gradient(to right,#ffffb2,#fecc5c,#fd8d3c,#f03b20,#bd0026)"></div>
  <div style="display:flex;justify-content:space-between;width:150px;margin-top:2px;color:#888;font-size:10px">
    <span>{min_grof}</span><span>{max_grof}</span>
  </div>

  <div style="margin:10px 0 6px;font-size:11px;color:#444;font-weight:600">Fijn</div>
  <div style="width:150px;height:12px;border-radius:3px;
    background:linear-gradient(to right,#ffffd9,#c7e9b4,#41b6c4,#2c7fb8,#253494)"></div>
  <div style="display:flex;justify-content:space-between;width:150px;margin-top:2px;color:#888;font-size:10px">
    <span>{min_fijn}</span><span>{max_fijn}</span>
  </div>

  <div style="margin-top:10px;color:#aaa;font-size:10px">Radius = vast (6px)<br>Kleur = lineaire telling</div>
</div>"""
m.get_root().html.add_child(folium.Element(legend_html))


# Save output html map
out_path = os.path.join(
    "../datasets/experiments/zwerfafval",
    f"heatmap_dots_{model}_{split}.html"
)
m.save(out_path)
print(f"Saved → {out_path}  ({len(points_wgs)} points)")

NameError: name 'counts_merged' is not defined

In [22]:
# With another format 'counts_260713.gpkg' as input for heatmap
GPKG_PATH = "/home/ding001/Zwerfafval-Detectie/datasets/experiments/zwerfafval/counts_260824.gpkg"

IMAGE_FOLDERS = [
    "/home/ding001/Zwerfafval-Detectie/datasets/experiments/zwerfafval/annotatieproject/inwinning_260824_images",
]

CACHE_PATH  = "/home/ding001/Zwerfafval-Detectie/datasets/experiments/zwerfafval/street_cache.json"
OUTPUT_PATH = "/home/ding001/Zwerfafval-Detectie/datasets/experiments/zwerfafval/heatmap_counts_260824.html"

gdf = gpd.read_file(GPKG_PATH, layer=0)
print(f"Total rows: {len(gdf)}")

# Filter out [selected] == False
gdf = gdf[gdf["selected"] == True].copy()
print(f"After selected=True filter: {len(gdf)} rows")

# Keep rows with  detection >1
gdf["total"] = gdf["Zwerfafval (fijn)"] + gdf["Zwerfafval (grof)"]
gdf = gdf[gdf["total"] > 0].copy()
print(f"After total > 0 filter: {len(gdf)} rows")

# Convert RD New (EPSG:28992) → WGS84 (EPSG:4326) for Folium
gdf_wgs = gdf.to_crs("EPSG:4326")

gdf_wgs.head()

Total rows: 11664
After selected=True filter: 4793 rows
After total > 0 filter: 3566 rows


,file_name,Zwerfafval (fijn),Zwerfafval (grof),timestamp_utc,straat_naam,dist_to_street,park_naam,dist_to_park,selected,section_mean_grof,section_mean_fijn,rolling_avg_grof,rolling_avg_fijn,geometry,total
0,20260824_111736_418914_000007,0.0,1.0,2026-08-24 11:17:36.418,Westhavenweg,9.623366,Westerpark,1110.317555,True,0.333333,0.0,1.509804,0.166667,POINT (4.85523 52.39675),1.0
58,20260824_113300_986516_012460,0.0,3.0,2026-08-24 11:33:00.986,Westhavenweg,5.640960,Westerpark,1104.206225,True,2.166667,0.5,1.065882,0.100000,POINT (4.85518 52.39666),3.0
64,20260824_113303_769282_012502,0.0,1.0,2026-08-24 11:33:03.769,Contactweg,7.303866,Westerpark,1098.711506,True,0.200000,0.0,0.999216,0.200000,POINT (4.85521 52.39661),1.0
78,20260824_113310_270181_012600,0.0,1.0,2026-08-24 11:33:10.270,Contactweg,8.369804,Westerpark,1083.894108,True,0.333333,0.0,0.253333,0.233333,POINT (4.85523 52.39646),1.0
90,20260824_113315_845830_012684,0.0,1.0,2026-08-24 11:33:15.845,Contactweg,7.929784,Westerpark,1066.578411,True,1.333333,0.0,0.666667,0.000000,POINT (4.85524 52.39626),1.0


In [ ]:
# Build a lookup index for images to avoid loading them from local every time
def build_image_index(folders):
    index = {}
    for folder in folders:
        if not os.path.isdir(folder):
            print(f"  ⚠ Folder not found: {folder}")
            continue
        for fname in os.listdir(folder):
            if fname.lower().endswith((".jpg", ".jpeg", ".png")):
                index[os.path.splitext(fname)[0]] = os.path.join(folder, fname)
    return index

image_index = build_image_index(IMAGE_FOLDERS)
print(f"Found {len(image_index)} images")


def get_image_b64(file_name):
    key  = os.path.splitext(os.path.basename(str(file_name)))[0]
    path = image_index.get(key)
    if path and os.path.isfile(path):
        with open(path, "rb") as f:
            data = base64.b64encode(f.read()).decode("utf-8")
        mime = "jpeg" if path.lower().endswith((".jpg", ".jpeg")) else "png"
        return f"data:image/{mime};base64,{data}"
    return None

Found 11667 images


In [ ]:
# Street name cache
street_cache = json.loads(open(CACHE_PATH).read()) if os.path.isfile(CACHE_PATH) else {}


def get_street_name(lat, lon):
    key = f"{lat:.5f},{lon:.5f}"
    if key in street_cache:
        return street_cache[key]
    try:
        r = requests.get(
            "https://nominatim.openstreetmap.org/reverse",
            params={"lat": lat, "lon": lon, "format": "json", "zoom": 17, "addressdetails": 1},
            headers={"User-Agent": "zwerfafval-heatmap/1.0"},
            timeout=5,
        )
        a = r.json().get("address", {}) if r.status_code == 200 else {}
        name = (
            a.get("road") or a.get("pedestrian")
            or a.get("footway") or a.get("cycleway") or "Onbekend"
        )
    except Exception:
        name = "Onbekend"
    street_cache[key] = name
    with open(CACHE_PATH, "w") as f:
        json.dump(street_cache, f)
    time.sleep(1.1)  # Nominatim rate limit: 1 req/sec
    return name


print(f"Fetching street names ({len(gdf_wgs)} points, {len(street_cache)} already cached)...")
street_names = {}
for i, row in enumerate(gdf_wgs.itertuples()):
    street_names[row.file_name] = get_street_name(row.geometry.y, row.geometry.x)
    if i % 20 == 0:
        print(f"  {i}/{len(gdf_wgs)}")
print("Done.")

Fetching street names (3566 points, 9372 already cached)...
  0/3566
  20/3566
  40/3566
  60/3566
  80/3566
  100/3566
  120/3566
  140/3566
  160/3566
  180/3566
  200/3566
  220/3566
  240/3566
  260/3566
  280/3566
  300/3566
  320/3566
  340/3566
  360/3566
  380/3566
  400/3566
  420/3566
  440/3566
  460/3566
  480/3566
  500/3566
  520/3566
  540/3566
  560/3566
  580/3566
  600/3566
  620/3566
  640/3566
  660/3566
  680/3566
  700/3566
  720/3566
  740/3566
  760/3566
  780/3566
  800/3566
  820/3566
  840/3566
  860/3566
  880/3566
  900/3566
  920/3566
  940/3566
  960/3566
  980/3566
  1000/3566
  1020/3566
  1040/3566
  1060/3566
  1080/3566
  1100/3566
  1120/3566
  1140/3566
  1160/3566
  1180/3566
  1200/3566
  1220/3566
  1240/3566
  1260/3566
  1280/3566
  1300/3566
  1320/3566
  1340/3566
  1360/3566
  1380/3566
  1400/3566
  1420/3566
  1440/3566
  1460/3566
  1480/3566
  1500/3566
  1520/3566
  1540/3566
  1560/3566
  1580/3566
  1600/3566
  1620/3566
  1640/3566


In [ ]:
# Generate color map
def make_to_hex(series, cmap_name):
    cmap = plt.get_cmap(cmap_name)
    vmin = series[series > 0].min()
    vmax = series.max()
    norm = mcolors.Normalize(vmin=vmin, vmax=max(vmax, 1))
    def to_hex(val):
        return mcolors.to_hex(cmap(norm(val)))
    return to_hex, int(vmin), round((vmin + vmax) / 2), int(vmax)

# Grof: green → orange → red  (YlOrRd starts too yellow, use custom)
from matplotlib.colors import LinearSegmentedColormap

grof_cmap = LinearSegmentedColormap.from_list(
    "grof", ["#2ecc71", "#e67e22", "#c0392b"]
)
fijn_cmap = LinearSegmentedColormap.from_list(
    "fijn",  ["#2ecc71", "#e67e22", "#c0392b"]
)

def make_to_hex_custom(series, cmap):
    vmin = series[series > 0].min()
    vmax = series.max()
    norm = mcolors.Normalize(vmin=vmin, vmax=max(vmax, 1))
    def to_hex(val):
        return mcolors.to_hex(cmap(norm(val)))
    return to_hex, int(vmin), round((vmin + vmax) / 2), int(vmax)

to_hex_grof, grof_min, grof_mid, grof_max = make_to_hex_custom(gdf_wgs["Zwerfafval (grof)"], grof_cmap)
to_hex_fijn, fijn_min, fijn_mid, fijn_max = make_to_hex_custom(gdf_wgs["Zwerfafval (fijn)"], fijn_cmap)

print(f"Grof scale: {grof_min} → {grof_mid} → {grof_max}")
print(f"Fijn scale: {fijn_min} → {fijn_mid} → {fijn_max}")

Grof scale: 1 → 6 → 12
Fijn scale: 1 → 10 → 19


In [ ]:
# Create Folium map with 2 layers (grof, fijn), and tooltips with images
center = [gdf_wgs.geometry.y.mean(), gdf_wgs.geometry.x.mean()]

m = folium.Map(location=center, zoom_start=13, tiles=None)
folium.TileLayer(
    tiles="https://t1.data.amsterdam.nl/topo_wm/{z}/{x}/{y}.png",
    attr="data.amsterdam.nl",
    name="Amsterdam Topo",
).add_to(m)

layer_grof = folium.FeatureGroup(name="Zwerfafval (grof)", show=True)
layer_fijn = folium.FeatureGroup(name="Zwerfafval (fijn)", show=True)

missing_img  = 0
GROF_THRESHOLD = 5  # image shown only if grof > treshold

def triangle_icon(color):
    """Smaller upward triangle, centered on the coordinate."""
    return folium.DivIcon(
        html=(
            f'<div style="'
            f'width:0;height:0;'
            f'border-left:5px solid transparent;'
            f'border-right:5px solid transparent;'
            f'border-bottom:9px solid {color};'
            f'filter:drop-shadow(0 0 1px rgba(0,0,0,0.6));'
            f'margin-top:-4px;margin-left:-5px'
            f'"></div>'
        ),
        icon_size=(10, 9),
        icon_anchor=(5, 4),
    )

def make_tooltip(street, dt_date, dt_time, grof, fijn, category, count, color, img_html=""):
    badge = (
        f'<span style="display:inline-block;background:{color};color:white;'
        f'border-radius:4px;padding:1px 7px;font-size:10px;font-weight:600;'
        f'margin-bottom:6px">{category}</span>'
    )
    return folium.Tooltip(
        f"""<div style="font-family:system-ui;font-size:12px;max-width:270px">
          {img_html}
          {badge}
          <table style="width:100%;border-collapse:collapse;line-height:1.8">
            <tr><td style="color:#888;padding-right:8px;white-space:nowrap">📍 Straat</td>
                <td style="font-weight:600;color:#111">{street}</td></tr>
            <tr><td style="color:#888;white-space:nowrap">📅 Datum</td>
                <td style="color:#444">{dt_date}</td></tr>
            <tr><td style="color:#888;white-space:nowrap">🕐 Tijd</td>
                <td style="color:#444">{dt_time}</td></tr>
            <tr><td style="color:#888;white-space:nowrap">📦 {category}</td>
                <td style="color:#444">{count} detecties</td></tr>
            <tr><td style="color:#888;white-space:nowrap">Totaal</td>
                <td style="color:#444">Grof {grof} &nbsp;|&nbsp; Fijn {fijn}</td></tr>
          </table>
        </div>""",
        sticky=True,
    )

for row in gdf_wgs.itertuples():
    lat   = row.geometry.y
    lon   = row.geometry.x
    grof  = int(row._3)
    fijn  = int(row._2)
    total = int(row.total)
    fname = str(row.file_name)

    ts = row.timestamp_utc
    if ts is not None and str(ts) != "NaT":
        dt_date = str(ts)[:10]
        dt_time = str(ts)[11:16]
    else:
        dt_date = dt_time = "Onbekend"

    street = street_names.get(fname, "Onbekend")

    # 'Grof' layer
    if grof > 0:
        color_grof = to_hex_grof(grof)

        if grof > GROF_THRESHOLD:
            # Triangle marker + image in tooltip
            img_b64 = get_image_b64(fname)
            if img_b64:
                img_html = (
                    f'<img src="{img_b64}" '
                    f'style="width:520px;height:340px;object-fit:cover;'
                    f'border-radius:5px;display:block;margin-bottom:8px">'
                )
            else:
                missing_img += 1
                img_html = (
                    f'<div style="width:520px;height:340px;background:#f0f0ee;border-radius:5px;'
                    f'display:flex;align-items:center;justify-content:center;'
                    f'color:#bbb;font-size:11px;margin-bottom:8px">Geen afbeelding</div>'
                )

            folium.Marker(
                location=[lat, lon],
                icon=triangle_icon(color_grof),
                tooltip=make_tooltip(street, dt_date, dt_time, grof, fijn, "Grof", grof, color_grof, img_html),
            ).add_to(layer_grof)

        else:
            # Small dot + pop-up, without image
            CircleMarker(
                location=[lat, lon],
                radius=4,
                color="#000000",      # black edge
                fill=True,
                fill_color=color_grof,
                fill_opacity=0.75,
                weight=0.3,             # edge thickness
                tooltip=make_tooltip(street, dt_date, dt_time, grof, fijn, "Grof", grof, color_grof),
            ).add_to(layer_grof)

    # 'Fijn' layer
    if fijn > 0:
        color_fijn = to_hex_fijn(fijn)
        CircleMarker(
            location=[lat, lon],
            radius=4,
            color=color_fijn,         # ← edge same as fill, invisible
            fill=True,
            fill_color=color_fijn,
            fill_opacity=0.6,
            weight=0.5,
            tooltip=make_tooltip(street, dt_date, dt_time, grof, fijn, "Fijn", fijn, color_fijn),
        ).add_to(layer_fijn)

layer_grof.add_to(m)
layer_fijn.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)

print(f"Triangles (grof > {GROF_THRESHOLD}): {sum(1 for r in gdf_wgs.itertuples() if int(r._3) > GROF_THRESHOLD)}")
print(f"Dots (grof 1–{GROF_THRESHOLD}): {sum(1 for r in gdf_wgs.itertuples() if 0 < int(r._3) <= GROF_THRESHOLD)}")
print(f"Missing images: {missing_img}")

Triangles (grof > 5): 106
Dots (grof 1–5): 2786
Missing images: 0


In [ ]:
# Create legend for linear color scales
legend_html = f"""
<div style="position:fixed;bottom:40px;left:20px;z-index:9999;
    background:white;border-radius:8px;padding:12px 16px;
    box-shadow:0 1px 6px rgba(0,0,0,.2);font-family:system-ui;font-size:12px">
  <div style="font-weight:600;margin-bottom:10px;color:#111">Detectiedichtheid</div>

  <div style="font-size:11px;font-weight:600;color:#444;margin-bottom:4px">Grof</div>
  <div style="width:150px;height:12px;border-radius:3px;
    background:linear-gradient(to right,#2ecc71,#e67e22,#c0392b)"></div>
  <div style="display:flex;justify-content:space-between;width:150px;margin-top:3px;color:#666;font-size:10px">
    <span>{grof_min}</span><span>{grof_mid}</span><span>{grof_max}</span>
  </div>

  <div style="font-size:11px;font-weight:600;color:#444;margin:12px 0 4px">Fijn</div>
  <div style="width:150px;height:12px;border-radius:3px;
    background:linear-gradient(to right,#2ecc71,#e67e22,#c0392b)"></div>
  <div style="display:flex;justify-content:space-between;width:150px;margin-top:3px;color:#666;font-size:10px">
    <span>{fijn_min}</span><span>{fijn_mid}</span><span>{fijn_max}</span>
  </div>
</div>"""
m.get_root().html.add_child(folium.Element(legend_html))

In [ ]:
# Save the final map to HTML
m.save(OUTPUT_PATH)
print(f"Saved → {OUTPUT_PATH}  ({os.path.getsize(OUTPUT_PATH)/1024/1024:.1f} MB)")
print(f"Points plotted: {len(gdf_wgs)}")

Saved → /home/ding001/Zwerfafval-Detectie/datasets/experiments/zwerfafval/heatmap_counts_260824.html  (99.5 MB)
Points plotted: 3566


In [ ]:
# Ranking most grof counts on streets for statstics

# Load & filter
gdf = gpd.read_file(GPKG_PATH, layer=0)
print(f"Total rows: {len(gdf)}")

# Filter out selected == False
gdf = gdf[gdf["selected"] == True].copy()
print(f"After selected=True filter: {len(gdf)} rows")

# Add total BEFORE converting CRS
gdf["total"] = gdf["Zwerfafval (fijn)"] + gdf["Zwerfafval (grof)"]
gdf = gdf[gdf["total"] > 0].copy()
print(f"After total > 0 filter: {len(gdf)} rows")

# Convert RD New → WGS84
gdf_wgs = gdf.to_crs("EPSG:4326")

# Verify columns and itertuples field mapping
print(f"\nColumns: {list(gdf_wgs.columns)}")
row0 = next(gdf_wgs.itertuples())
print(f"itertuples fields: {row0._fields}")

gdf_wgs.head()

Total rows: 11664
After selected=True filter: 4793 rows
After total > 0 filter: 3566 rows

Columns: ['file_name', 'Zwerfafval (fijn)', 'Zwerfafval (grof)', 'timestamp_utc', 'straat_naam', 'dist_to_street', 'park_naam', 'dist_to_park', 'selected', 'section_mean_grof', 'section_mean_fijn', 'rolling_avg_grof', 'rolling_avg_fijn', 'geometry', 'total']
itertuples fields: ('Index', 'file_name', '_2', '_3', 'timestamp_utc', 'straat_naam', 'dist_to_street', 'park_naam', 'dist_to_park', 'selected', 'section_mean_grof', 'section_mean_fijn', 'rolling_avg_grof', 'rolling_avg_fijn', 'geometry', 'total')


,file_name,Zwerfafval (fijn),Zwerfafval (grof),timestamp_utc,straat_naam,dist_to_street,park_naam,dist_to_park,selected,section_mean_grof,section_mean_fijn,rolling_avg_grof,rolling_avg_fijn,geometry,total
0,20260824_111736_418914_000007,0.0,1.0,2026-08-24 11:17:36.418,Westhavenweg,9.623366,Westerpark,1110.317555,True,0.333333,0.0,1.509804,0.166667,POINT (4.85523 52.39675),1.0
58,20260824_113300_986516_012460,0.0,3.0,2026-08-24 11:33:00.986,Westhavenweg,5.640960,Westerpark,1104.206225,True,2.166667,0.5,1.065882,0.100000,POINT (4.85518 52.39666),3.0
64,20260824_113303_769282_012502,0.0,1.0,2026-08-24 11:33:03.769,Contactweg,7.303866,Westerpark,1098.711506,True,0.200000,0.0,0.999216,0.200000,POINT (4.85521 52.39661),1.0
78,20260824_113310_270181_012600,0.0,1.0,2026-08-24 11:33:10.270,Contactweg,8.369804,Westerpark,1083.894108,True,0.333333,0.0,0.253333,0.233333,POINT (4.85523 52.39646),1.0
90,20260824_113315_845830_012684,0.0,1.0,2026-08-24 11:33:15.845,Contactweg,7.929784,Westerpark,1066.578411,True,1.333333,0.0,0.666667,0.000000,POINT (4.85524 52.39626),1.0


In [31]:
# Load street cache
street_cache = json.loads(open(CACHE_PATH).read()) if os.path.isfile(CACHE_PATH) else {}
print(f"Cache entries: {len(street_cache)}")

# Round coordinates to 5 decimals to match this heatmap notebook
gdf_wgs["lat_r"] = gdf_wgs.geometry.y.round(5)
gdf_wgs["lon_r"] = gdf_wgs.geometry.x.round(5)

gdf_wgs["street"] = gdf_wgs.apply(
    lambda row: street_cache.get(f"{row.lat_r:.5f},{row.lon_r:.5f}", "Onbekend"),
    axis=1
)

uncached = (gdf_wgs["street"] == "Onbekend").sum()
print(f"Uncached points (Onbekend): {uncached}")
print(f"Matched points: {len(gdf_wgs) - uncached}")


Cache entries: 9372
Uncached points (Onbekend): 10
Matched points: 3556


In [32]:
# Aggregate & rank
top20 = (
    gdf_wgs.groupby("street")["Zwerfafval (grof)"]
    .agg(
        **{"Grof (totaal)": "sum",
           "Aantal punten": "count",
           "Gem. grof per foto": "mean"}
    )
    .sort_values("Grof (totaal)", ascending=False)
    .head(20)
    .reset_index()
    .rename(columns={"street": "Straat"})
)

# Round average to 1 decimal
top20["Gem. grof per foto"] = top20["Gem. grof per foto"].round(1)

top20.index += 1  # rank starts at 1

top20

,Straat,Grof (totaal),Aantal punten,Gem. grof per foto
1,Spaarndammerdijk,336.0,192,1.8
2,Brettenpad,302.0,220,1.4
3,Tasmanstraat,301.0,162,1.9
4,Nieuwe Hemweg,293.0,158,1.9
5,Contactweg,289.0,104,2.8
6,Jan van Galenstraat,267.0,132,2.0
7,Van Diemenstraat,265.0,136,1.9
8,Zaanstraat,247.0,116,2.1
9,Spaarndammerstraat,245.0,168,1.5
10,Transformatorweg,172.0,134,1.3
